# Matched-pair recall-gap test using fine-grained features (CUB-200-2011)

We want to measure whether an attribute probe is truly using visual evidence for the attribute,
or whether it is partially relying on species identity as a shortcut.

Core idea:
For a given attribute and a pair of species (S1, S2), we build a matched test set where
attribute prevalence is identical in both species:
- same number of positive examples in S1 and S2
- same number of negative examples in S1 and S2

Then we evaluate:
- recall on positives for S1
- recall on positives for S2
- the recall gap |recall(S1) - recall(S2)|

If the recall gap is consistently large even after perfect prevalence matching,
that suggests the probe is using species-specific cues, not just attribute evidence.

We run this across:
- many attributes
- many species pairs
- multiple random seeds (because subsampling is random)
and summarize the recall gaps.


For each attribute:

1. Train a linear probe on top of frozen visual features.
2. Evaluate the probe on held-out test images.
3. Group test images by species.
4. For pairs of species:
   - Subsample images so that both species have the same number of
     attribute-positive and attribute-negative examples.
   - Compute recall on attribute-positive images for each species.
5. Measure the recall gap between species.

In [1]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn


In [2]:
ROOT = Path("/scratch/network/cr7998/cv_emergence_project")
CUB  = ROOT / "data" / "CUB_200_2011"

ATTR_TXT = ROOT / "data" / "attributes.txt"   # attr_id -> attr_name like has_primary_color::yellow

BASE_FEAT = ROOT / "features" / "resnet50_cub_fine"
CBM_FEAT  = ROOT / "features" / "resnet50_cub_cbm_fine"

assert CUB.exists(), f"Missing CUB folder: {CUB}"
assert ATTR_TXT.exists(), f"Missing attributes.txt: {ATTR_TXT}"
assert BASE_FEAT.exists(), f"Missing baseline fine features: {BASE_FEAT}"
assert CBM_FEAT.exists(), f"Missing cbm fine features: {CBM_FEAT}"

device = "cuda" if torch.cuda.is_available() else "cpu"
device


'cpu'

In [3]:
# LF-CBM: we will probe LF-CBM *concept activations* proj_c (after W_c + normalization),
# but KEEP your CUB attribute labels as y (same as baseline/CBM).

LFCBM_REPO = Path("/scratch/network/cr7998/Label-free-CBM")

# Set this to the LF-CBM model folder you trained (the folder that contains W_c.pt, W_g.pt, b_g.pt, proj_mean.pt, proj_std.pt, args.txt)
# Example: LFCBM_MODEL_DIR = LFCBM_REPO / "saved_models" / "cub_lf_cbm"
LFCBM_MODEL_DIR = LFCBM_REPO / "saved_models" / "cub_lf_cbm"

# Where we will save extracted per-image concept activations (proj_c) in *your* project format
LFCBM_FEAT = ROOT / "features" / "resnet50_cub_lfcbm_proj"

print("LFCBM_REPO:", LFCBM_REPO)
print("LFCBM_MODEL_DIR:", LFCBM_MODEL_DIR)
print("LFCBM_FEAT:", LFCBM_FEAT)

assert LFCBM_REPO.exists(), f"Missing LF-CBM repo: {LFCBM_REPO}"
assert LFCBM_MODEL_DIR.exists(), f"Missing LF-CBM model dir: {LFCBM_MODEL_DIR}"

LFCBM_REPO: /scratch/network/cr7998/Label-free-CBM
LFCBM_MODEL_DIR: /scratch/network/cr7998/Label-free-CBM/saved_models/cub_lf_cbm
LFCBM_FEAT: /scratch/network/cr7998/cv_emergence_project/features/resnet50_cub_lfcbm_proj


In [4]:
# LF-CBM: we will probe LF-CBM *concept activations* proj_c (after W_c + normalization),
# but KEEP your CUB attribute labels as y (same as baseline/CBM).

LFCBM_REPO = Path("/scratch/network/cr7998/Label-free-CBM")

# Set this to the LF-CBM model folder you trained (the folder that contains W_c.pt, W_g.pt, b_g.pt, proj_mean.pt, proj_std.pt, args.txt)
# Example: LFCBM_MODEL_DIR = LFCBM_REPO / "saved_models" / "cub_lf_cbm"
LFCBM_MODEL_DIR = LFCBM_REPO / "saved_models" / "cub_lf_cbm"

# Where we will save extracted per-image concept activations (proj_c) in *your* project format
LFCBM_FEAT = ROOT / "features" / "resnet50_cub_lfcbm_proj"

print("LFCBM_REPO:", LFCBM_REPO)
print("LFCBM_MODEL_DIR:", LFCBM_MODEL_DIR)
print("LFCBM_FEAT:", LFCBM_FEAT)

assert LFCBM_REPO.exists(), f"Missing LF-CBM repo: {LFCBM_REPO}"
assert LFCBM_MODEL_DIR.exists(), f"Missing LF-CBM model dir: {LFCBM_MODEL_DIR}"

LFCBM_REPO: /scratch/network/cr7998/Label-free-CBM
LFCBM_MODEL_DIR: /scratch/network/cr7998/Label-free-CBM/saved_models/cub_lf_cbm
LFCBM_FEAT: /scratch/network/cr7998/cv_emergence_project/features/resnet50_cub_lfcbm_proj


In [5]:
def load_species_maps(cub_root: Path):
    """
    Loads species ID to name mappings from classes.txt.
    Also produces a prettified version for printing.
    """
    classes = pd.read_csv(
        cub_root / "classes.txt",
        sep=r"\s+",
        header=None,
        names=["species_id", "class_name"],
        engine="python"
    )

    def pretty(name: str) -> str:
        # Example: "001.Black_footed_Albatross" -> "Black footed albatross"
        return name.split(".", 1)[-1].replace("_", " ")

    id_to_pretty = {
        int(r.species_id): pretty(r.class_name)
        for _, r in classes.iterrows()
    }

    return id_to_pretty

species_id_to_name = load_species_maps(CUB)

def spname(sid: int) -> str:
    return species_id_to_name.get(int(sid), f"species_{sid}")


In [6]:
def load_meta(cub_root: Path) -> pd.DataFrame:
    """
    Returns a dataframe mapping each image to:
    - species ID
    - train/test split
    """
    img_species = pd.read_csv(
        cub_root / "image_class_labels.txt",
        sep=r"\s+",
        header=None,
        names=["image_id", "species_id"],
        engine="python"
    )

    split_df = pd.read_csv(
        cub_root / "train_test_split.txt",
        sep=r"\s+",
        header=None,
        names=["image_id", "is_train"],
        engine="python"
    )

    meta = img_species.merge(split_df, on="image_id")
    meta["species_name"] = meta["species_id"].map(spname)
    return meta

meta = load_meta(CUB)


In [7]:
def load_image_attr_labels_robust(cub_root: Path) -> pd.DataFrame:
    path = cub_root / "attributes" / "image_attribute_labels.txt"
    rows = []
    bad = 0

    with open(path, "r") as f:
        for line in f:
            toks = line.strip().split()
            if len(toks) < 4:
                bad += 1
                continue
            try:
                image_id = int(toks[0])
                attr_id  = int(toks[1])
                is_pres  = int(toks[2])
                cert     = int(toks[3])
                rows.append((image_id, attr_id, is_pres, cert))
            except:
                bad += 1

    df = pd.DataFrame(rows, columns=["image_id", "attr_id", "is_present", "certainty"])
    print("Parsed rows:", len(df), "bad lines skipped:", bad)
    return df

img_attr_long = load_image_attr_labels_robust(CUB)
img_attr_long.head()


Parsed rows: 3677856 bad lines skipped: 0


,image_id,attr_id,is_present,certainty
0,1,1,0,3
1,1,2,0,3
2,1,3,0,3
3,1,4,0,3
4,1,5,1,3


In [8]:
def load_attr_maps(attr_txt: Path):
    rows = []
    with open(attr_txt, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            aid_str, name = line.split(" ", 1)
            rows.append((int(aid_str), name))
    df = pd.DataFrame(rows, columns=["attr_id", "attr_name"])
    name_to_id = dict(zip(df["attr_name"], df["attr_id"]))
    id_to_name = dict(zip(df["attr_id"], df["attr_name"]))
    return df, name_to_id, id_to_name

attr_df, attr_name_to_id, attr_id_to_name = load_attr_maps(ATTR_TXT)
attr_df.head()


,attr_id,attr_name
0,1,has_bill_shape::curved_(up_or_down)
1,2,has_bill_shape::dagger
2,3,has_bill_shape::hooked
3,4,has_bill_shape::needle
4,5,has_bill_shape::hooked_seabird


In [9]:
ATTR_LIST = [
    "has_primary_color::yellow",
    "has_throat_color::yellow",
    "has_underparts_color::yellow",
    "has_belly_color::yellow",
    "has_breast_color::yellow",
]
for a in ATTR_LIST:
    assert a in attr_name_to_id, f"Missing attribute in attributes.txt: {a}"


In [10]:
# What this cell does:
# - For a given attribute_id, merges:
#   meta (image->species, split) with attribute labels (image->y)
# - Produces a clean table with y in {0,1}. where y is whether the attribut below is present or not.
# Why it matters:
# - This is the ground-truth label table used for training and evaluation.

def build_attr_labeled_df(meta: pd.DataFrame,
                          img_attr_long: pd.DataFrame,
                          attr_id: int,
                          min_certainty: int = 1) -> pd.DataFrame:
    """
    Returns dataframe with:
      image_id, species_id, species_name, is_train, y, certainty
    Only keeps annotations with certainty >= min_certainty.
    """
    sub = img_attr_long[img_attr_long["attr_id"] == int(attr_id)].copy()
    sub = sub[sub["certainty"] >= int(min_certainty)].copy()

    out = meta.merge(sub[["image_id", "is_present", "certainty"]], on="image_id", how="inner")
    out = out.rename(columns={"is_present": "y"})
    out["y"] = out["y"].astype(int)
    return out[["image_id", "species_id", "species_name", "is_train", "y", "certainty"]]

print("Defined:", "build_attr_labeled_df")

# sanity check on one attribute
attr_name = "has_primary_color::yellow"
aid = attr_name_to_id[attr_name]
lab = build_attr_labeled_df(meta, img_attr_long, aid, min_certainty=1)
print("Attribute:", attr_name, "rows labeled:", len(lab), "pos rate:", lab.y.mean())
lab.head()


Defined: build_attr_labeled_df
Attribute: has_primary_color::yellow rows labeled: 11788 pos rate: 0.1506616898540889


,image_id,species_id,species_name,is_train,y,certainty
0,1,1,Black footed Albatross,0,0,3
1,2,1,Black footed Albatross,1,0,4
2,3,1,Black footed Albatross,0,0,4
3,4,1,Black footed Albatross,1,0,4
4,5,1,Black footed Albatross,1,0,4


In [11]:
# - Loads a feature tensor from disk and converts it to float32 torch.Tensor.

def safe_torch_load(path: Path):
    """
    Uses weights_only=True if supported to reduce pickle risk warnings.
    """
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        return torch.load(path, map_location="cpu")

def load_features(feat_dir: Path, layer: str, split: str) -> torch.Tensor:
    """
    Loads feature tensor saved as {layer}_{split}.pt from feat_dir.
    """
    p = feat_dir / f"{layer}_{split}.pt"
    assert p.exists(), f"Missing: {p}"
    X = safe_torch_load(p)
    if not isinstance(X, torch.Tensor):
        X = torch.tensor(X)
    return X.float()

In [27]:
import pickle
import torch
import numpy as np

def to_1d_int_array(x):
    """Convert tensor/list/np array to 1D int numpy array."""
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().numpy()
    x = np.array(x)
    x = x.reshape(-1)
    return x.astype(int)


def load_split_order(feat_dir, split):
    p = feat_dir / f"labels_{split}.pt"
    assert p.exists(), f"Missing: {p}"

    # labels_*.pt often stores numpy arrays / dicts and isn't "weights-only" compatible.
    try:
        t = torch.load(p, map_location="cpu", weights_only=True)
    except (pickle.UnpicklingError, RuntimeError, TypeError):
        # Only do this if you trust the source (you generated it).
        t = torch.load(p, map_location="cpu", weights_only=False)

    assert isinstance(t, dict), f"Expected dict in {p}, got {type(t)}"

    # be flexible about key names
    key = None
    for k in ["image_ids", "img_ids", "ids", "image_id", "imageid"]:
        if k in t:
            key = k
            break
    assert key is not None, f"{p} missing image-id key; has {list(t.keys())}"

    ids = to_1d_int_array(t[key])
    kind = infer_kind(ids)
    return kind, ids


def infer_kind(arr):
    # Heuristic:
    # - species ids: 1..200 (sometimes 0..199)
    # - image ids: 1..11788
    if arr.max() <= 200 and arr.min() >= 0:
        return "species_id_like"
    if arr.max() > 200:
        return "image_id_like"
    return "unknown"


In [30]:
LFCBM_FEAT  = Path("/scratch/network/cr7998/cv_emergence_project/features/lfcbm_cub")  # adjust to exact folder
LFCBM_LAYER = "proj_c"

base_kind_tr, base_ids_tr = load_split_order(BASE_FEAT, "train")
base_kind_te, base_ids_te = load_split_order(BASE_FEAT, "test")

cbm_kind_tr,  cbm_ids_tr  = load_split_order(CBM_FEAT, "train")
cbm_kind_te,  cbm_ids_te  = load_split_order(CBM_FEAT, "test")

lf_kind_tr,   lf_ids_tr   = load_split_order(LFCBM_FEAT, "train")
lf_kind_te,   lf_ids_te   = load_split_order(LFCBM_FEAT, "test")

print("Baseline:", base_kind_tr, (base_ids_tr.min(), base_ids_tr.max()), "|", base_kind_te, (base_ids_te.min(), base_ids_te.max()))
print("CBM     :", cbm_kind_tr,  (cbm_ids_tr.min(), cbm_ids_tr.max()),   "|", cbm_kind_te,  (cbm_ids_te.min(), cbm_ids_te.max()))
print("LF-CBM  :", lf_kind_tr,   (lf_ids_tr.min(), lf_ids_tr.max()),     "|", lf_kind_te,   (lf_ids_te.min(), lf_ids_te.max()))

assert lf_kind_tr == "image_id_like" and lf_kind_te == "image_id_like", \
    "LF-CBM labels_{split}.pt must contain image_ids for alignment."

Baseline: image_id_like (2, 11787) | image_id_like (1, 11788)
CBM     : image_id_like (2, 11787) | image_id_like (1, 11788)
LF-CBM  : image_id_like (2, 11787) | image_id_like (1, 11788)


In [14]:
# What this cell does:
# - Aligns features (in feature row order) to attribute labels by image_id.
# - Produces X_aligned and df_aligned with the same ordering.


def align_features_and_labels(X_split: torch.Tensor,
                              image_ids_in_feature_order: np.ndarray,
                              labeled_df_split: pd.DataFrame):
    """
    Inputs:
      X_split: feature tensor of shape [N, D]
      image_ids_in_feature_order: length N, image_id for each row of X_split
      labeled_df_split: dataframe with at least columns [image_id, y, species_id, species_name]

    Output:
      X_aligned: features for images that have labels
      df_aligned: same rows, same order, includes y and species info
    """
    labeled = labeled_df_split.set_index("image_id")[["y", "species_id", "species_name"]]

    keep_idx = []
    rows = []
    for i, img_id in enumerate(image_ids_in_feature_order):
        img_id = int(img_id)
        if img_id in labeled.index:
            keep_idx.append(i)
            y, sid, sname = labeled.loc[img_id]
            rows.append((img_id, int(sid), str(sname), int(y)))

    X_aligned = X_split[keep_idx]
    df_aligned = pd.DataFrame(rows, columns=["image_id", "species_id", "species_name", "y"])
    return X_aligned, df_aligned

In [15]:
# What this cell does:
# - Defines a linear probe (single linear layer).
# - Trains it using BCEWithLogitsLoss with mild class-imbalance handling.
# Why it matters:
# - Probe is the measurement instrument for "is the attribute encoded in features?"

class LinearProbe(nn.Module):
    def __init__(self, d: int):
        super().__init__()
        self.lin = nn.Linear(d, 1)

    def forward(self, x):
        return self.lin(x).squeeze(-1)

def train_probe(Xtr: torch.Tensor, ytr: np.ndarray,
                seed=0, lr=1e-2, wd=1e-4, epochs=25, batch=512):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    Xtr = Xtr.to(device)
    ytr_t = torch.tensor(ytr, dtype=torch.float32, device=device)

    probe = LinearProbe(Xtr.shape[1]).to(device)
    opt = torch.optim.AdamW(probe.parameters(), lr=lr, weight_decay=wd)

    pos = float(ytr_t.mean().item())
    pos_weight = torch.tensor([(1 - pos) / pos], device=device) if 0 < pos < 1 else torch.tensor([1.0], device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    n = Xtr.shape[0]
    for _ in range(epochs):
        perm = torch.randperm(n, device=device)
        for i in range(0, n, batch):
            idx = perm[i:i+batch]
            logits = probe(Xtr[idx])
            loss = loss_fn(logits, ytr_t[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()

    return probe

@torch.no_grad()
def predict_probs(probe: nn.Module, X: torch.Tensor, batch=4096) -> np.ndarray:
    probe.eval()
    probs = []
    for i in range(0, X.shape[0], batch):
        xb = X[i:i+batch].to(device)
        logits = probe(xb)
        probs.append(torch.sigmoid(logits).detach().cpu())
    return torch.cat(probs, dim=0).numpy()

print("Defined:", "LinearProbe", "train_probe", "predict_probs")


Defined: LinearProbe train_probe predict_probs


1. Identify species that have both positive and negative examples.
2. For each pair:
   - Subsample so both species have identical numbers of positives and negatives.
3. Compute recall on positive examples for each species.
4. Measure the absolute recall difference.

In [16]:
# What this cell does:
# - Finds species that have enough positives and negatives for the chosen attribute.
# - Samples many species pairs.
# - For each pair, subsamples to match prevalence exactly and computes recall on positives.
# Why it matters:
# - This isolates species-specific differences even when prevalence is controlled perfectly.

def make_candidate_pairs(df_test: pd.DataFrame, min_each=10, max_pairs=200, seed=0):
    """
    Returns list of tuples (sid_A, sid_B, mpos, mneg) where:
      mpos = min(posA, posB)
      mneg = min(negA, negB)
    and both are >= min_each.
    """
    g = df_test.groupby("species_id")["y"].agg(["count", "sum"]).rename(columns={"sum": "pos"})
    g["neg"] = g["count"] - g["pos"]
    ok = g[(g["pos"] >= min_each) & (g["neg"] >= min_each)]
    sids = ok.index.to_list()

    rng = np.random.default_rng(seed)
    pairs = []
    if len(sids) < 2:
        return pairs

    for _ in range(max_pairs * 10):
        a, b = rng.choice(sids, size=2, replace=False)
        mpos = int(min(ok.loc[a, "pos"], ok.loc[b, "pos"]))
        mneg = int(min(ok.loc[a, "neg"], ok.loc[b, "neg"]))
        if mpos >= min_each and mneg >= min_each:
            pairs.append((int(a), int(b), mpos, mneg))
        if len(pairs) >= max_pairs:
            break
    return pairs

def matched_pair_eval(df_test: pd.DataFrame, probs: np.ndarray, sid_A: int, sid_B: int,
                      mpos: int, mneg: int, seed=0, thr=0.5):
    """
    Subsamples to:
      mpos positives + mneg negatives from each species.
    Computes recall on positive examples only for each species subset.
    """
    df = df_test.copy()
    df["prob"] = probs

    A = df[df.species_id == sid_A]
    B = df[df.species_id == sid_B]

    A_pos, A_neg = A[A.y == 1], A[A.y == 0]
    B_pos, B_neg = B[B.y == 1], B[B.y == 0]

    A_s = pd.concat([A_pos.sample(mpos, random_state=seed), A_neg.sample(mneg, random_state=seed)])
    B_s = pd.concat([B_pos.sample(mpos, random_state=seed), B_neg.sample(mneg, random_state=seed)])

    def recall_pos(d):
        pos = d[d.y == 1]
        pred = (pos.prob.values >= thr).astype(int)
        return float((pred == 1).mean()) if len(pos) else np.nan

    recA = recall_pos(A_s)
    recB = recall_pos(B_s)

    return {
        "sid_A": sid_A,
        "sid_B": sid_B,
        "species_A": spname(sid_A),
        "species_B": spname(sid_B),
        "npos": int(mpos),
        "nneg": int(mneg),
        "recall_A": float(recA),
        "recall_B": float(recB),
        "gap": float(abs(recA - recB)),
    }

def species_recall_prevalence_table(df_te: pd.DataFrame, probs: np.ndarray, thr=0.5) -> pd.DataFrame:
    """
    Per-species table on the TEST set:
      - n, n_pos, n_neg
      - prevalence = n_pos / n
      - tp = # of positives predicted positive
      - recall = tp / n_pos
      - (optional) precision if you want it later
    """
    df = df_te[["species_id", "species_name", "y"]].copy()
    df["prob"] = probs
    df["pred"] = (df["prob"] >= thr).astype(int)

    # Basic counts per species
    g = (df.groupby(["species_id", "species_name"], as_index=False)
           .agg(
               n=("y", "size"),
               n_pos=("y", "sum"),
               n_pred_pos=("pred", "sum"),
           ))
    g["n_neg"] = g["n"] - g["n_pos"]
    g["prevalence"] = g["n_pos"] / g["n"]

    # True positives per species (only among y==1)
    tp = (df[df["y"] == 1]
            .groupby(["species_id", "species_name"])["pred"]
            .sum()
            .reset_index(name="tp"))

    out = g.merge(tp, on=["species_id", "species_name"], how="left")
    out["tp"] = out["tp"].fillna(0).astype(int)

    # Recall: tp / n_pos (handle n_pos==0)
    out["recall"] = np.where(out["n_pos"] > 0, out["tp"] / out["n_pos"], np.nan)

    # Nice sorting: show biggest sample sizes first (or sort by recall if you prefer)
    out = out.sort_values(["n"], ascending=False).reset_index(drop=True)
    return out



In [17]:
import numpy as np

def bootstrap_ci(x, alpha=0.05):
    x = np.asarray(x, dtype=float)
    return (float(np.quantile(x, alpha/2)),
            float(np.quantile(x, 1 - alpha/2)))

def bootstrap_p_value(x):
    x = np.asarray(x, dtype=float)
    p_lo = float(np.mean(x <= 0))
    p_hi = float(np.mean(x >= 0))
    return 2.0 * min(p_lo, p_hi)


In [18]:
# Runs the full pipeline for ONE attribute and ONE model's features:
# 1) Build labeled train/test sets for this attribute (with certainty filtering)
# 2) Load train/test features for the chosen layer
# 3) Align features to labels by image_id using split order arrays
# 4) Train a linear probe on train features
# 5) Predict probabilities on test features
# 6) Generate candidate species pairs that allow matched sampling
# 7) For each pair and each seed, compute matched-pair recall gap
# 8) Summarize gaps per pair (mean/std across seeds) and compute aggregate stats

def run_one_attribute(
    attr_name: str,
    feat_dir: Path,
    split_order_kind_train: str,
    split_order_train: np.ndarray,
    split_order_kind_test: str,
    split_order_test: np.ndarray,
    layer: str,
    *,
    min_certainty: int = 1,
    thr: float = 0.5,
    epochs: int = 25,
    min_each: int = 10,
    n_pairs: int = 200,
    seeds=(0, 1, 2),
):
    """
    Run ONE attribute on ONE model's features:
      - train a linear probe (train split)
      - predict probabilities (test split)
      - compute per-species prevalence + recall table (test split)
      - optionally run matched-pair recall-gap test (if n_pairs > 0)
    """

    # Alignment requires feature order indexed by image_id.
    assert split_order_kind_train == "image_id_like" and split_order_kind_test == "image_id_like", (
        "Cannot align features to attribute labels because labels_{split}.pt is not image_id-like.\n"
        "If you hit this, we need to read the dataset ordering from your extractor code."
    )

    # Build per-image labels for this attribute (with certainty filtering)
    aid = attr_name_to_id[attr_name]
    lab = build_attr_labeled_df(meta, img_attr_long, aid, min_certainty=min_certainty)

    lab_train = lab[lab["is_train"] == 1].copy()
    lab_test  = lab[lab["is_train"] == 0].copy()

    # Load precomputed features for this layer/split
    Xtr_all = load_features(feat_dir, layer, "train")
    Xte_all = load_features(feat_dir, layer, "test")

    # Align labeled images to feature tensor order
    Xtr, df_tr = align_features_and_labels(Xtr_all, split_order_train, lab_train)
    Xte, df_te = align_features_and_labels(Xte_all, split_order_test,  lab_test)

    ytr = df_tr["y"].astype(int).to_numpy()
    yte = df_te["y"].astype(int).to_numpy()

    # Train probe + predict probabilities on test
    probe = train_probe(Xtr, ytr, seed=0, epochs=epochs)
    probs = predict_probs(probe, Xte)

    # Per-species prevalence + recall on test (this is what your prof cares about)
    species_table = species_recall_prevalence_table(df_te, probs, thr=thr)

    # Overall test accuracy at chosen threshold
    test_acc = float(((probs >= thr).astype(int) == yte).mean()) if len(yte) else np.nan

    # ---------- Matched pairs (optional) ----------
    matched_cols = [
        "sid_A", "sid_B", "species_A", "species_B",
        "npos", "nneg", "recall_A", "recall_B", "gap"
    ]
    summary_cols = ["sid_A", "sid_B", "species_A", "species_B", "gap_mean", "gap_std", "n_runs", "npos", "nneg"]

    if n_pairs is None or int(n_pairs) <= 0:
        res = pd.DataFrame(columns=matched_cols)
        pair_summary = pd.DataFrame(columns=summary_cols)
        pairs = []
    else:
        pairs = make_candidate_pairs(df_te, min_each=min_each, max_pairs=n_pairs, seed=0)

        rows = []
        for (a, b, mpos, mneg) in pairs:
            for sd in seeds:
                rows.append(matched_pair_eval(df_te, probs, a, b, mpos, mneg, seed=sd, thr=thr))

        res = pd.DataFrame(rows, columns=matched_cols)

        if res.empty:
            pair_summary = pd.DataFrame(columns=summary_cols)
        else:
            pair_summary = (
                res.groupby(["sid_A","sid_B","species_A","species_B"], as_index=False)
                   .agg(
                       gap_mean=("gap","mean"),
                       gap_std=("gap","std"),
                       n_runs=("gap","size"),
                       npos=("npos","min"),
                       nneg=("nneg","min"),
                       gap_ci_lo=("gap", lambda x: bootstrap_ci(x)[0]),
                       gap_ci_hi=("gap", lambda x: bootstrap_ci(x)[1]),
                       gap_p=("gap", bootstrap_p_value),
                   )
            )
            EPS = 1e-12

            # 1) SNR: big gap that is also stable across seeds
            pair_summary["gap_snr"] = pair_summary["gap_mean"] / (pair_summary["gap_std"].fillna(0.0) + EPS)
            
            # 2) Normalized-by-uncertainty (more meaningful than dividing by 1.0)
            #    uses CI width as uncertainty estimate; smaller CI => larger normalized effect
            pair_summary["gap_ci_width"] = pair_summary["gap_ci_hi"] - pair_summary["gap_ci_lo"]

            # 0..1 scale “how big is the gap?” (this is already the meaning of gap_mean)
            pair_summary["gap_norm"] = pair_summary["gap_mean"]
            
            # “uncertainty-normalized effect size” (can be >1, often much larger)
            pair_summary["gap_u"] = pair_summary["gap_mean"] / (pair_summary["gap_ci_width"] + EPS)

                        

    
    # Aggregate gap stats (only meaningful if matched rows exist)
    mean_gap = float(res["gap"].mean()) if ("gap" in res.columns and len(res)) else np.nan
    p90_gap  = float(res["gap"].quantile(0.9)) if ("gap" in res.columns and len(res)) else np.nan

    info = {
        "attr": attr_name,
        "layer": layer,
        "n_train": int(len(df_tr)),
        "n_test": int(len(df_te)),
        "train_pos_rate": float(ytr.mean()) if len(ytr) else np.nan,
        "test_pos_rate": float(yte.mean()) if len(yte) else np.nan,
        "test_acc": float(test_acc),
        "thr": float(thr),
        "epochs": int(epochs),
        "n_pairs": int(len(pairs)),
        "mean_gap": mean_gap,
        "p90_gap": p90_gap,
    }

    return info, res, pair_summary, df_te, species_table

In [19]:
def screen_attributes_for_species_variation(
    candidate_attrs,
    feat_dir: Path,
    kind_tr: str, ids_tr: np.ndarray,
    kind_te: str, ids_te: np.ndarray,
    layer: str,
    *,
    min_certainty: int = 1,
    thr: float = 0.5,
    min_pos_per_species: int = 10,
    min_species_with_pos: int = 15, #recheck
    min_overall_prev: float = 0.05,
    max_overall_prev: float = 0.95,
    epochs: int = 8,
    max_attrs: int | None = None,
    verbose_every: int = 50,
    keep_error_examples: int = 5,
):
    rows = []
    errors = []
    stats = {
        "tried": 0,
        "success": 0,
        "filtered_too_few_species_pos": 0,
        "filtered_prev_out_of_range": 0,
        "filtered_no_recall_vals": 0,
        "errored": 0,
    }

    cand = list(candidate_attrs)
    if max_attrs is not None:
        cand = cand[:max_attrs]

    for i, attr in enumerate(cand):
        stats["tried"] += 1
        try:
            info, _, _, _, species_table = run_one_attribute(
                attr,
                feat_dir,
                kind_tr, ids_tr,
                kind_te, ids_te,
                layer=layer,
                min_certainty=min_certainty,
                min_each=10,
                n_pairs=0,          # screening: no matched pairs
                seeds=(0,),
                epochs=epochs,
                thr=thr,
            )

            st = species_table.copy()
            overall_prev = float(st["n_pos"].sum() / st["n"].sum()) if st["n"].sum() > 0 else np.nan

            st_pos = st[st["n_pos"] >= min_pos_per_species].copy()
            n_species_pos = int(len(st_pos))

            if n_species_pos < min_species_with_pos:
                stats["filtered_too_few_species_pos"] += 1
                continue

            if not (min_overall_prev <= overall_prev <= max_overall_prev):
                stats["filtered_prev_out_of_range"] += 1
                continue

            recall_vals = st_pos["recall"].dropna().to_numpy()
            if recall_vals.size == 0:
                stats["filtered_no_recall_vals"] += 1
                continue

            stats["success"] += 1

            recall_std = float(np.std(recall_vals))
            recall_range = float(np.max(recall_vals) - np.min(recall_vals))
            recall_p90_p10 = float(np.quantile(recall_vals, 0.9) - np.quantile(recall_vals, 0.1))

            rows.append({
                "attr": attr,
                "overall_prev": overall_prev,
                "n_species_pos": n_species_pos,
                "recall_std": recall_std,
                "recall_range": recall_range,
                "recall_p90_p10": recall_p90_p10,
                "test_acc": float(info["test_acc"]),
                "n_test": int(info["n_test"]),
            })

            if verbose_every and (i % verbose_every == 0):
                print(f"[{i}/{len(cand)}] ok: {attr}  prev={overall_prev:.3f}  n_species_pos={n_species_pos}")

        except Exception as e:
            stats["errored"] += 1
            if len(errors) < keep_error_examples:
                errors.append((attr, repr(e)))
            continue

    screen_df = pd.DataFrame(rows)

    print("\n--- Screening summary ---")
    for k, v in stats.items():
        print(f"{k}: {v}")
    if errors:
        print("\nExample errors (first few):")
        for a, msg in errors:
            print(" ", a, "->", msg)

    if screen_df.empty:
        print("\nNo attributes passed filters. Likely causes:")
        print(" - run_one_attribute is erroring for most attrs (see errors above)")
        print(" - filters too strict for your attribute distribution")
        return screen_df

    screen_df = screen_df.sort_values(
        ["recall_p90_p10", "recall_range", "recall_std"],
        ascending=False
    ).reset_index(drop=True)

    return screen_df


# ---- Run it ----
CANDIDATE_ATTRS = attr_df["attr_name"].tolist()

screen_df = screen_attributes_for_species_variation(
    CANDIDATE_ATTRS,
    feat_dir=BASE_FEAT,
    kind_tr=base_kind_tr, ids_tr=base_ids_tr,
    kind_te=base_kind_te, ids_te=base_ids_te,
    layer=LAYER,
    min_certainty=1,
    thr=0.5,
    min_pos_per_species=10,
    min_species_with_pos=15,
    min_overall_prev=0.05,
    max_overall_prev=0.95,
    epochs=8,
    max_attrs=200,      # IMPORTANT: start small so you see errors quickly
    verbose_every=25,
)

if screen_df.empty:
    # Loosen constraints automatically so you get *something*
    screen_df = screen_attributes_for_species_variation(
        CANDIDATE_ATTRS,
        feat_dir=BASE_FEAT,
        kind_tr=base_kind_tr, ids_tr=base_ids_tr,
        kind_te=base_kind_te, ids_te=base_ids_te,
        layer=LAYER,
        min_certainty=1,
        thr=0.5,
        min_pos_per_species=5,
        min_species_with_pos=8,
        min_overall_prev=0.02,
        max_overall_prev=0.98,
        epochs=6,
        max_attrs=200,
        verbose_every=25,
    )

if not screen_df.empty:
    TOP_K = 12
    ATTR_LIST = screen_df["attr"].head(TOP_K).tolist()
    print("\nNew ATTR_LIST:")
    for a in ATTR_LIST:
        print(" ", a)
    screen_df.head(20)


# Map: attr -> typical cross-species recall spread (from screening)
# This is the scale we normalize gaps against.
attr_to_spread = screen_df.set_index("attr")["recall_p90_p10"].to_dict()


[25/200] ok: has_upperparts_color::brown  prev=0.239  n_species_pos=59
[50/200] ok: has_underparts_color::black  prev=0.177  n_species_pos=39
[75/200] ok: has_tail_shape::notched_tail  prev=0.296  n_species_pos=81
[100/200] ok: has_head_pattern::eyering  prev=0.160  n_species_pos=20
[125/200] ok: has_throat_color::grey  prev=0.171  n_species_pos=30

--- Screening summary ---
tried: 200
success: 73
filtered_too_few_species_pos: 127
filtered_prev_out_of_range: 0
filtered_no_recall_vals: 0
errored: 0

New ATTR_LIST:
  has_upper_tail_color::white
  has_bill_shape::all-purpose
  has_breast_pattern::solid
  has_upperparts_color::black
  has_bill_length::shorter_than_head
  has_upperparts_color::brown
  has_wing_color::black
  has_breast_color::white
  has_wing_color::grey
  has_wing_color::white
  has_under_tail_color::black
  has_under_tail_color::white


In [31]:
# Runs the matched-pair pipeline across multiple attributes (ATTR_LIST)
# for both models:
# - baseline features (BASE_FEAT)
# - CBM features (CBM_FEAT)
#
# run_many(...) loops attributes, calls run_one_attribute(...),
# stores:
# - info_df: per-attribute run metadata (acc, mean gap, etc.)
# - pairs_df: per-(attr, pair) gap_mean/gap_std results
#
# This produces baseline_info/baseline_pairs and cbm_info/cbm_pairs
# for downstream comparison and reporting.

#   Meaning of printed numbers:
#     - test_acc: test-set accuracy of the attribute probe/classifier for this attribute
#                 (on this model's features at the chosen layer)
#     - mean_gap: average matched-pair recall gap across the sampled species pairs
#                 for this attribute (higher = more species-dependent / entangled)

# Choose a fine layer to use consistently
LAYER = "layer4.0"

def run_many(attr_list, model_name, feat_dir, kind_tr, ids_tr, kind_te, ids_te,
             layer, min_certainty=1, min_each=10, n_pairs=200, seeds=(0,1,2), thr=0.5, epochs=25):

    all_info = []
    all_pair_summ = []
    all_species_tables = []

    for attr in attr_list:
        info, res, pair_summ, df_te, species_table = run_one_attribute(
            attr, feat_dir,
            kind_tr, ids_tr,
            kind_te, ids_te,
            layer=layer,
            min_certainty=min_certainty,
            min_each=min_each,
            n_pairs=n_pairs,
            seeds=seeds,
            thr=thr,
            epochs=epochs,
        )

        info["model"] = model_name
        all_info.append(info)

        pair_summ = pair_summ.copy()
        pair_summ["attr"] = attr
        pair_summ["model"] = model_name
        all_pair_summ.append(pair_summ)

        species_table = species_table.copy()
        species_table["attr"] = attr
        species_table["model"] = model_name
        all_species_tables.append(species_table)

        print(model_name, attr, "test_acc=", round(info["test_acc"], 4), "mean_gap=", round(info["mean_gap"], 4))

    info_df = pd.DataFrame(all_info)
    pairs_df = pd.concat(all_pair_summ, ignore_index=True) if all_pair_summ else pd.DataFrame()
    species_df = pd.concat(all_species_tables, ignore_index=True) if all_species_tables else pd.DataFrame()
    return info_df, pairs_df, species_df


In [32]:
# Run the full analysis for both models:
# For each model, run_many returns:
# 1) info_df     : per-attribute summary stats (accuracy, mean gap, etc.)
# 2) pairs_df    : matched-pair recall gap results (controlled evaluation)
# 3) species_df  : per-species prevalence + recall table (overall evaluation)

# ---- Code Cell 21 ----
# Choose layers per model
BASE_LAYER = "layer4.0"
CBM_LAYER  = "layer4.0"
LFCBM_LAYER = "proj_c"

baseline_info, baseline_pairs, baseline_species = run_many(
    ATTR_LIST,
    model_name="baseline",
    feat_dir=BASE_FEAT,
    kind_tr=base_kind_tr, ids_tr=base_ids_tr,
    kind_te=base_kind_te, ids_te=base_ids_te,
    layer=BASE_LAYER,
    min_certainty=1,
    min_each=10,
    n_pairs=200,
    seeds=(0, 1, 2),
    thr=0.5,
    epochs=25,
)

cbm_info, cbm_pairs, cbm_species = run_many(
    ATTR_LIST,
    model_name="cbm",
    feat_dir=CBM_FEAT,
    kind_tr=cbm_kind_tr, ids_tr=cbm_ids_tr,
    kind_te=cbm_kind_te, ids_te=cbm_ids_te,
    layer=CBM_LAYER,
    min_certainty=1,
    min_each=10,
    n_pairs=200,
    seeds=(0, 1, 2),
    thr=0.5,
    epochs=25,
)

lfcbm_info, lfcbm_pairs, lfcbm_species = run_many(
    ATTR_LIST,
    model_name="lfcbm",
    feat_dir=LFCBM_FEAT,
    kind_tr=lf_kind_tr, ids_tr=lf_ids_tr,
    kind_te=lf_kind_te, ids_te=lf_ids_te,
    layer=LFCBM_LAYER,
    min_certainty=1,
    min_each=10,
    n_pairs=200,
    seeds=(0, 1, 2),
    thr=0.5,
    epochs=25,
)

baseline_info, cbm_info, lfcbm_info



baseline has_upper_tail_color::white test_acc= 0.8631 mean_gap= 0.3909
baseline has_bill_shape::all-purpose test_acc= 0.6593 mean_gap= 0.3344
baseline has_breast_pattern::solid test_acc= 0.6748 mean_gap= 0.2938
baseline has_upperparts_color::black test_acc= 0.6921 mean_gap= 0.3102
baseline has_bill_length::shorter_than_head test_acc= 0.7316 mean_gap= 0.3144
baseline has_upperparts_color::brown test_acc= 0.7468 mean_gap= 0.2169
baseline has_wing_color::black test_acc= 0.632 mean_gap= 0.2338
baseline has_breast_color::white test_acc= 0.7401 mean_gap= 0.2108
baseline has_wing_color::grey test_acc= 0.6729 mean_gap= 0.3074
baseline has_wing_color::white test_acc= 0.7527 mean_gap= 0.2704
baseline has_under_tail_color::black test_acc= 0.593 mean_gap= 0.1805
baseline has_under_tail_color::white test_acc= 0.7333 mean_gap= 0.2007
cbm has_upper_tail_color::white test_acc= 0.7839 mean_gap= 0.2466
cbm has_bill_shape::all-purpose test_acc= 0.694 mean_gap= 0.3118
cbm has_breast_pattern::solid test_ac

(                                  attr     layer  n_train  n_test  \
 0          has_upper_tail_color::white  layer4.0     5994    5794   
 1          has_bill_shape::all-purpose  layer4.0     5994    5794   
 2            has_breast_pattern::solid  layer4.0     5994    5794   
 3          has_upperparts_color::black  layer4.0     5994    5794   
 4   has_bill_length::shorter_than_head  layer4.0     5994    5794   
 5          has_upperparts_color::brown  layer4.0     5994    5794   
 6                has_wing_color::black  layer4.0     5994    5794   
 7              has_breast_color::white  layer4.0     5994    5794   
 8                 has_wing_color::grey  layer4.0     5994    5794   
 9                has_wing_color::white  layer4.0     5994    5794   
 10         has_under_tail_color::black  layer4.0     5994    5794   
 11         has_under_tail_color::white  layer4.0     5994    5794   
 
     train_pos_rate  test_pos_rate  test_acc  thr  epochs  n_pairs  mean_gap  \
 0      

In [33]:
baseline_species.to_csv("baseline_species.csv", index=False)
cbm_species.to_csv("cbm_species.csv", index=False)
lfcbm_species.to_csv("lfcbm_species.csv", index=False)

Pairs from below

In [34]:
# Collapses the per-(attr, species pair) table into an attribute-level summary:
# For each (model, attr), computes:
# - gap_mean: average gap_mean across all species pairs for that attribute
# - gap_max: the maximum gap_mean pair (worst disparity) for that attribute
# - n_pairs: number of evaluated species pairs for that attribute
#
# Then concatenates baseline + cbm summaries into one table for easy comparison.

def summarize_by_attr(pairs_df: pd.DataFrame):
    if pairs_df.empty:
        return pairs_df

    g = pairs_df.groupby(["model", "attr"], as_index=False)

    out = g.agg(
        gap_mean=("gap_mean","mean"),
        gap_median=("gap_mean","median"),
        gap_max=("gap_mean","max"),
        n_pairs=("gap_mean","size"),
        frac_ci_above0=("gap_ci_lo", lambda s: float(np.mean(np.asarray(s) > 0))),
        frac_p_small=("gap_p", lambda s: float(np.mean(np.asarray(s) <= 0.05))),
    ).sort_values(["model", "gap_mean"], ascending=[True, False])

    return out



summary = pd.concat([
    summarize_by_attr(baseline_pairs),
    summarize_by_attr(cbm_pairs),
    summarize_by_attr(lfcbm_pairs),
], ignore_index=True)

summary


,model,attr,gap_mean,gap_median,gap_max,n_pairs,frac_ci_above0,frac_p_small
0,baseline,has_upper_tail_color::white,0.381963,0.388889,0.833333,173,0.913295,0.832370
1,baseline,has_bill_shape::all-purpose,0.331824,0.300000,1.000000,198,0.924242,0.873737
2,baseline,has_bill_length::shorter_than_head,0.309857,0.254902,1.000000,195,0.953846,0.871795
3,baseline,has_upperparts_color::black,0.308617,0.266667,0.911111,198,0.964646,0.898990
4,baseline,has_wing_color::grey,0.305030,0.264286,0.900000,198,0.909091,0.853535
5,baseline,has_breast_pattern::solid,0.293778,0.222222,0.866667,200,0.940000,0.850000
6,baseline,has_wing_color::white,0.272800,0.233333,0.800000,194,0.855670,0.788660
7,baseline,has_wing_color::black,0.232983,0.200000,0.846154,199,0.859296,0.768844
8,baseline,has_upperparts_color::brown,0.218025,0.138095,0.727273,196,0.913265,0.821429
9,baseline,has_breast_color::white,0.212422,0.194444,0.800000,197,0.857868,0.751269


In [35]:
def add_gap_interpretability_cols(pair_df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds interpretability columns to a *pair_summary* dataframe:
      - gap_snr  = gap_mean / gap_std  (higher = more stable across bootstrap runs)
      - gap_norm = gap_mean / max_possible_gap

    max_possible_gap explanation (why this is a reasonable scale):
      For a fixed threshold thr, prevalence p = P(y=1) puts a hard upper bound on recall differences.
      If the classifier predicts positive on fraction r = P(pred=1), then:
        recall = P(pred=1 | y=1) <= min(1, r/p)
      So across two species with prevalences pA, pB (estimated in matched subset as mpos/(mpos+mneg)),
      the absolute recall gap is bounded by:
        max_gap <= |min(1, r/pA) - min(1, r/pB)|
      We approximate r by the threshold under a well-calibrated model as ~thr (rough heuristic),
      but we can instead just use a safe upper bound:
        max_gap <= 1.0
      To avoid pretending we know r exactly, we use a simple *prevalence-only* bound:
        max_gap_prevalence = 1.0  (conservative)
      and keep gap_norm mainly as “gap_mean on a 0..1 scale”.

    If you later want a tighter bound, pass in the actual per-species predicted-positive rate.
    """
    out = pair_df.copy()

    # Stability: mean gap relative to its bootstrap variability.
    out["gap_snr"] = out["gap_mean"] / out["gap_std"].replace(0, np.nan)

    # Matched prevalence within each species subset is the same by construction:
    # prevalence_matched = mpos / (mpos + mneg)
    # This isn't the *dataset* prevalence, but it's the prevalence of the evaluation slice.
    prev_matched = out["npos"] / (out["npos"] + out["nneg"])
    out["prev_matched"] = prev_matched.astype(float)

    # Conservative normalization (0..1 scale). This avoids overclaiming a "true" max gap.
    out["gap_norm"] = out["gap_mean"] / 1.0

    return out

### Interpreting matched-pair gap columns

Each row corresponds to a *species pair* evaluated for a single attribute and model.

- **gap_mean**  
  Mean absolute difference in recall between the two species, averaged over repeated
  matched-pair resampling runs.  
  *This is the raw recall gap.*

- **gap_std**  
  Standard deviation of the recall gap across repeated runs with different random seeds.  
  *Measures how stable the gap estimate is.*

- **gap_norm**  
  `gap_mean` normalized by the attribute’s typical cross-species recall spread
  (defined as the 90th–10th percentile recall difference across species).  
  *(Is this pair’s gap large relative to how much this attribute usually varies
  across species?)*  
  Values near 1 indicate an extreme pair; values near 0 indicate negligible disparity.

- **gap_snr**  
  Signal-to-noise ratio of the gap: `gap_mean / gap_std`.  
  *Answers: “Is the gap consistently observed, or within noise?”*  
  Larger values indicate a stable, repeatable gap.

- **npos / nneg**  
  Number of positive and negative examples per species used in each matched subset.  
  *Ensures both species are compared under equal prevalence.*

- **n_runs**  
  Number of matched-pair resampling runs used to estimate the gap.  
  *Higher values increase confidence in `gap_mean` and `gap_std`.*

Overall, **gap_norm** indicates *magnitude* (how large the disparity is),
while **gap_snr** indicates *reliability* (how confident we are it is not noise).


In [36]:
# Utility to display the "worst" (largest gap_mean) species pairs for a given attribute and model:
# Filters to (model, attr), sorts by gap_mean descending, prints the top-k pairs.

def top_pairs(pairs_df: pd.DataFrame, model: str, attr: str, k=10):
    """
    Filters to (model, attr), sorts by gap_mean descending, returns top-k pairs.

    Interpreting new columns:
      - gap_ci_lo / gap_ci_hi: bootstrap CI over matched resamples (seeds). If CI excludes 0 => stable gap.
      - gap_p: bootstrap p-value for H0: gap==0 (two-sided).
      - gap_snr: mean gap / std gap (higher => more stable across resamples).
      - gap_norm: gap_mean on 0..1 scale (currently conservative; 0.2 = 20 percentage-point recall gap).
    """
    sub = pairs_df[(pairs_df["model"] == model) & (pairs_df["attr"] == attr)].copy()
    if sub.empty:
        return sub

    cols = [
      "species_A","species_B",
      "gap_mean","gap_ci_lo","gap_ci_hi","gap_p",
      "gap_std","gap_snr","gap_norm","gap_u",
      "npos","nneg","n_runs"
    ]

    # keep only columns that exist (safe if you run old cached tables)
    cols = [c for c in cols if c in sub.columns]

    return sub.sort_values("gap_mean", ascending=False).head(k)[["species_A","species_B","gap_mean","gap_ci_lo","gap_ci_hi","gap_p", 
                                                                 "gap_std","gap_snr","gap_norm","npos","nneg","n_runs"]]



for a in ATTR_LIST:
    print("\nAttribute:", a)
    print("Baseline top pairs:")
    display(top_pairs(baseline_pairs, "baseline", a, k=10))
    print("CBM top pairs:")
    display(top_pairs(cbm_pairs, "cbm", a, k=10))
    print("LF-CBM top pairs:")
    display(top_pairs(lfcbm_pairs, "lfcbm", a, k=10))




Attribute: has_upper_tail_color::white
Baseline top pairs:


,species_A,species_B,gap_mean,gap_ci_lo,gap_ci_hi,gap_p,gap_std,gap_snr,gap_norm,npos,nneg,n_runs
102,Caspian Tern,Blue Jay,0.833333,0.800,0.895000,0.0,0.057735,1.443376e+01,0.833333,10,17,3
104,Caspian Tern,Clark Nutcracker,0.833333,0.800,0.895000,0.0,0.057735,1.443376e+01,0.833333,10,17,3
70,Red legged Kittiwake,Blue Jay,0.800000,0.800,0.800000,0.0,0.000000,8.000000e+11,0.800000,10,12,3
56,Blue Jay,Red legged Kittiwake,0.800000,0.800,0.800000,0.0,0.000000,8.000000e+11,0.800000,10,12,3
71,Red legged Kittiwake,Clark Nutcracker,0.800000,0.800,0.800000,0.0,0.000000,8.000000e+11,0.800000,10,12,6
78,Clark Nutcracker,Elegant Tern,0.800000,0.800,0.800000,0.0,0.000000,8.000000e+11,0.800000,10,18,3
12,American Goldfinch,Artic Tern,0.777778,0.750,0.829167,0.0,0.048113,1.616581e+01,0.777778,12,14,3
124,Elegant Tern,Downy Woodpecker,0.777778,0.750,0.829167,0.0,0.048113,1.616581e+01,0.777778,12,17,3
54,Blue Jay,Western Gull,0.766667,0.705,0.800000,0.0,0.057735,1.327906e+01,0.766667,10,15,3
36,Herring Gull,Clark Nutcracker,0.766667,0.705,0.800000,0.0,0.057735,1.327906e+01,0.766667,10,11,3


CBM top pairs:


,species_A,species_B,gap_mean,gap_ci_lo,gap_ci_hi,gap_p,gap_std,gap_snr,gap_norm,npos,nneg,n_runs
54,Blue Jay,Western Gull,0.700000,0.700,0.700,0.0,0.000000,7.000000e+11,0.700000,10,15,3
56,Blue Jay,Red legged Kittiwake,0.700000,0.700,0.700,0.0,0.000000,7.000000e+11,0.700000,10,12,3
136,Black and white Warbler,Blue Jay,0.700000,0.700,0.700,0.0,0.000000,7.000000e+11,0.700000,10,16,3
70,Red legged Kittiwake,Blue Jay,0.700000,0.700,0.700,0.0,0.000000,7.000000e+11,0.700000,10,12,3
102,Caspian Tern,Blue Jay,0.666667,0.605,0.700,0.0,0.057735,1.154701e+01,0.666667,10,17,3
55,Blue Jay,Pied Kingfisher,0.666667,0.600,0.700,0.0,0.051640,1.290994e+01,0.666667,10,11,6
64,Pied Kingfisher,Blue Jay,0.666667,0.605,0.700,0.0,0.057735,1.154701e+01,0.666667,10,11,3
53,Blue Jay,Ring billed Gull,0.666667,0.605,0.700,0.0,0.057735,1.154701e+01,0.666667,10,18,3
58,Blue Jay,Forsters Tern,0.633333,0.600,0.695,0.0,0.057735,1.096966e+01,0.633333,10,14,3
51,Western Gull,Black Tern,0.600000,0.600,0.600,0.0,0.000000,6.000000e+11,0.600000,10,15,3


LF-CBM top pairs:


...

,species_A,species_B,gap_mean,gap_ci_lo,gap_ci_hi,gap_p,gap_std,gap_snr,gap_norm,npos,nneg,n_runs
2272,Elegant Tern,Cactus Wren,0.600000,0.600000,0.600000,0.0,0.000000,6.000000e+11,0.600000,10,13,3
2324,Cactus Wren,Black and white Warbler,0.600000,0.600000,0.600000,0.0,0.000000,6.000000e+11,0.600000,10,11,3
2223,Clark Nutcracker,Ring billed Gull,0.583333,0.583333,0.583333,0.0,0.000000,5.833333e+11,0.583333,12,10,6
2258,Artic Tern,Cactus Wren,0.566667,0.505000,0.600000,0.0,0.057735,9.814955e+00,0.566667,10,10,3
2321,Cactus Wren,Artic Tern,0.566667,0.505000,0.600000,0.0,0.057735,9.814955e+00,0.566667,10,10,3
2253,Artic Tern,Clark Nutcracker,0.527778,0.500000,0.579167,0.0,0.048113,1.096966e+01,0.527778,12,10,3
2222,Clark Nutcracker,Herring Gull,0.527778,0.500000,0.579167,0.0,0.048113,1.096966e+01,0.527778,12,10,3
2323,Cactus Wren,Forsters Tern,0.500000,0.500000,0.500000,0.0,0.000000,5.000000e+11,0.500000,10,11,3
2318,Cactus Wren,Western Gull,0.500000,0.500000,0.500000,0.0,0.000000,5.000000e+11,0.500000,10,12,3
2319,Cactus Wren,Pied Kingfisher,0.500000,0.500000,0.500000,0.0,0.000000,5.000000e+11,0.500000,10,17,3


CBM top pairs:


,species_A,species_B,gap_mean,gap_ci_lo,gap_ci_hi,gap_p,gap_std,gap_snr,gap_norm,npos,nneg,n_runs
2201,Ring billed Gull,Chipping Sparrow,0.333333,0.333333,0.333333,0.0,0.0,3.333333e+11,0.333333,12,10,3
2214,Pied Kingfisher,Chipping Sparrow,0.333333,0.333333,0.333333,0.0,0.0,3.333333e+11,0.333333,12,17,3
2219,Red legged Kittiwake,Chipping Sparrow,0.333333,0.333333,0.333333,0.0,0.0,3.333333e+11,0.333333,12,11,3
2244,Chipping Sparrow,Red legged Kittiwake,0.333333,0.333333,0.333333,0.0,0.0,3.333333e+11,0.333333,12,11,3
2243,Chipping Sparrow,Ring billed Gull,0.333333,0.333333,0.333333,0.0,0.0,3.333333e+11,0.333333,12,10,3
2319,Cactus Wren,Pied Kingfisher,0.300000,0.300000,0.300000,0.0,0.0,3.000000e+11,0.300000,10,17,3
2318,Cactus Wren,Western Gull,0.300000,0.300000,0.300000,0.0,0.0,3.000000e+11,0.300000,10,12,3
2326,Cactus Wren,Red cockaded Woodpecker,0.300000,0.300000,0.300000,0.0,0.0,3.000000e+11,0.300000,10,17,3
2321,Cactus Wren,Artic Tern,0.300000,0.300000,0.300000,0.0,0.0,3.000000e+11,0.300000,10,10,3
2323,Cactus Wren,Forsters Tern,0.300000,0.300000,0.300000,0.0,0.0,3.000000e+11,0.300000,10,11,3


LF-CBM top pairs:


,species_A,species_B,gap_mean,gap_ci_lo,gap_ci_hi,gap_p,gap_std,gap_snr,gap_norm,npos,nneg,n_runs
2245,Chipping Sparrow,Forsters Tern,0.666667,0.666667,0.666667,0.0,0.0,6.666667e+11,0.666667,12,11,3
2243,Chipping Sparrow,Red legged Kittiwake,0.666667,0.666667,0.666667,0.0,0.0,6.666667e+11,0.666667,12,11,3
2200,Ring billed Gull,Chipping Sparrow,0.666667,0.666667,0.666667,0.0,0.0,6.666667e+11,0.666667,12,10,3
2242,Chipping Sparrow,Ring billed Gull,0.666667,0.666667,0.666667,0.0,0.0,6.666667e+11,0.666667,12,10,3
2213,Pied Kingfisher,Chipping Sparrow,0.666667,0.666667,0.666667,0.0,0.0,6.666667e+11,0.666667,12,17,3
2218,Red legged Kittiwake,Chipping Sparrow,0.666667,0.666667,0.666667,0.0,0.0,6.666667e+11,0.666667,12,11,3
2299,Red bellied Woodpecker,Chipping Sparrow,0.666667,0.666667,0.666667,0.0,0.0,6.666667e+11,0.666667,12,15,3
2247,Chipping Sparrow,Red bellied Woodpecker,0.666667,0.666667,0.666667,0.0,0.0,6.666667e+11,0.666667,12,15,3
2248,Chipping Sparrow,Downy Woodpecker,0.666667,0.666667,0.666667,0.0,0.0,6.666667e+11,0.666667,12,14,3
2241,Chipping Sparrow,Glaucous winged Gull,0.583333,0.583333,0.583333,0.0,0.0,5.833333e+11,0.583333,12,12,3


### How confidence intervals and p-values are computed

For each species pair and attribute, we run the matched-pair evaluation multiple times
(`n_runs`, using different random seeds).  
Each run produces one recall gap value.  
These gap values form an *empirical sampling distribution* of the recall gap.

#### Bootstrap confidence interval (CI)

- We treat the list of gap values from the repeated matched resampling runs as a
  bootstrap distribution.
- The **95% confidence interval** is computed using the *percentile method*:
  - `gap_ci_lo` = 2.5th percentile of the gap values
  - `gap_ci_hi` = 97.5th percentile of the gap values
- Interpretation:
  - If the interval **excludes 0**, the recall gap is stable under resampling.
  - Narrow intervals indicate low sampling variability; wide intervals indicate
    uncertainty due to limited data or few runs.

#### Bootstrap p-value

- We test the null hypothesis **H₀: recall gap = 0**.
- The p-value is computed directly from the empirical gap distribution:
  - Compute the fraction of runs where the gap is ≤ 0
  - Compute the fraction of runs where the gap is ≥ 0
  - The two-sided p-value is  
    `p = 2 × min(P(gap ≤ 0), P(gap ≥ 0))`
- This p-value measures how often the resampled gaps are consistent with no difference
  in recall between the two species.

- A gap is considered **statistically significant** if:
  - `gap_p ≤ 0.05`, **and**
  - the confidence interval `[gap_ci_lo, gap_ci_hi]` does not include `0`
- In practice, we also check effect size:
  - very small gaps can be statistically significant with enough resamples, but are
    not substantively meaningful
  - therefore, significance is interpreted jointly with `gap_mean` and `gap_snr`

**Important note:**  
The bootstrap distribution here reflects variability induced by *matched resampling*,
not independent image-level noise. This avoids parametric assumptions that do not hold
in fine-grained species datasets.


What to do next


### 1. Compare layers relative to species emergence

Run the same matched-pair test at:
- layer3.x
- layer4.0
- avgpool

If recall gaps increase after the species-emergence layer, this supports the idea that
species representations are feeding back into attribute prediction?



### 2. Aggregate across attributes

Instead of looking at single attributes:
- Compute mean gap across all attributes
- Measure fraction of pairs with gap > 0.3

This shows whether entanglement is a general phenomenon or limited to color attributes.



### 3. Negative control attributes

Test attributes that should be weakly species-correlated (e.g. rare shapes).

Why:
If gaps shrink for these attributes, it strengthens the causal interpretation
that species identity drives
